# Create the train datastracture for 3 edges

In [1]:
import pandas as pd

# Specify the features to keep
features_to_keep = [
    'Src IP', 'Dst IP', 'Timestamp', 'Label', 
    'Src Port', 'Dst Port', 'Fwd Header Len', 'Init Bwd Win Byts', 
    'Fwd Seg Size Avg', 'Fwd Pkt Len Mean', 'Init Fwd Win Byts', 
    'Fwd Pkt Len Max', 'TotLen Fwd Pkts', 'Bwd Pkt Len Mean', 
    'Idle Min', 'Bwd Header Len', 'Pkt Len Var', 'Subflow Fwd Byts', 
    'TotLen Bwd Pkts', 'Idle Max', 'Fwd Seg Size Min', 'Idle Mean', 
    'Pkt Len Max', 'Bwd Pkt Len Std', 'Bwd Pkt Len Max', 
    'Protocol', 'Pkt Len Mean', 'Down/Up Ratio'
]

# Load your dataset
data = pd.read_csv('train_data.csv')  # Replace 'your_data.csv' with your actual file name

# Keep only the specified features
filtered_data = data[features_to_keep]

# Convert 'Timestamp' to datetime
filtered_data['Timestamp'] = pd.to_datetime(filtered_data['Timestamp'])

# Order the data by 'Timestamp'
filtered_data = filtered_data.sort_values(by='Timestamp')

# Save the temporally ordered data to a new file
filtered_data.to_csv('filtered_train_3edge.csv', index=False)

print("Filtered and temporally ordered data saved to 'filtered_train_3edge.csv'.")


Filtered and temporally ordered data saved to 'filtered_train_3edge.csv'.


In [2]:
#check for inside of csv (just for test, no need for run)
import pandas as pd

# Load the CSV file
file_path = "filtered_train_3edge.csv"  # Replace with your actual file path
df = pd.read_csv(file_path)

# Check if the label column contains '1'
label_column = 'Label'  # Replace with the actual label column name if different
if label_column in df.columns:
    label_distribution = df[label_column].value_counts()
    print("Label Distribution:")
    print(label_distribution)

    if 1 in label_distribution.index:
        print("The CSV contains label '1'.")
    else:
        print("The CSV does NOT contain label '1'.")
else:
    print(f"'{label_column}' column not found in the CSV.")

Label Distribution:
Label
0    100000
Name: count, dtype: int64
The CSV does NOT contain label '1'.


# created hourly graph with 3 edges from train dataset

In [3]:
import pandas as pd
import networkx as nx
import os
import pickle  # Added for saving graphs

def create_test_graphs_edge_labels(df, output_dir):
    """
    Split the DataFrame into hourly slices and create graphs for each slice.
    Each edge gets a valid label (e.g., 0 or 1) read from the DataFrame.
    
    Parameters:
        df (pd.DataFrame): The input DataFrame with temporal data.
        output_dir (str): Directory to save the graphs.
    """
    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)
    
    # Group the DataFrame into hourly slices using the datetime index.
    # Changed 'H' to 'h' to resolve the pandas FutureWarning
    time_slices = [g for _, g in df.groupby(pd.Grouper(freq='h'))]
    
    for slice_index, slice_df in enumerate(time_slices):
        if slice_df.empty:
            continue

        # Print value counts of the 'Label' column in this time-slice.
        print(f"Hour {slice_index}:")
        print(slice_df['Label'].value_counts())
        
        # Create a MultiDiGraph for this time-slice.
        G = nx.MultiDiGraph()

        for _, row in slice_df.iterrows():
            src_ip = row['Src IP']
            dst_ip = row['Dst IP']
            
            # Convert the label to an int (if missing or invalid, you can decide a fallback; here we assume it is valid)
            try:
                label = int(row['Label'])
            except Exception as e:
                print(f"Skipping row due to invalid label: {row['Label']}; error: {e}")
                continue

            if pd.isna(src_ip) or pd.isna(dst_ip):
                continue

            # Add nodes if not already present.
            if not G.has_node(src_ip):
                G.add_node(src_ip)
            if not G.has_node(dst_ip):
                G.add_node(dst_ip)

            # Add edges for different interactions.

            # 1. Network Edge (Communication stats)
            G.add_edge(src_ip, dst_ip, key='network', label=label,
                       src_port=row['Src Port'], dst_port=row['Dst Port'], 
                       protocol=row['Protocol'], pkt_len_mean=row['Pkt Len Mean'], 
                       pkt_len_max=row['Pkt Len Max'], pkt_len_var=row['Pkt Len Var'],
                       bwd_pkt_len_mean=row['Bwd Pkt Len Mean'], bwd_pkt_len_std=row['Bwd Pkt Len Std'],
                       bwd_pkt_len_max=row['Bwd Pkt Len Max'], fwd_pkt_len_mean=row['Fwd Pkt Len Mean'],
                       fwd_pkt_len_max=row['Fwd Pkt Len Max'], tot_len_fwd=row['TotLen Fwd Pkts'],
                       tot_len_bwd=row['TotLen Bwd Pkts'], down_up_ratio=row['Down/Up Ratio'],
                       interaction='network_communication')

            # 2. Context Edge (Timing and environment)
            G.add_edge(src_ip, dst_ip, key='context', label=label,
                       idle_min=row['Idle Min'], idle_max=row['Idle Max'], idle_mean=row['Idle Mean'],
                       fwd_seg_size_avg=row['Fwd Seg Size Avg'], fwd_seg_size_min=row['Fwd Seg Size Min'],
                       subflow_fwd_byts=row['Subflow Fwd Byts'],
                       interaction='context')

            # 3. Knowledge Edge (Protocol/Header rules)
            G.add_edge(src_ip, dst_ip, key='knowledge', label=label,
                       fwd_header_len=row['Fwd Header Len'], bwd_header_len=row['Bwd Header Len'],
                       init_fwd_win=row['Init Fwd Win Byts'], init_bwd_win=row['Init Bwd Win Byts'],
                       interaction='knowledge')

        # Save the graph as a .gpickle file using standard pickle
        graph_path = os.path.join(output_dir, f"test_graph_hour_{slice_index}.gpickle")
        with open(graph_path, 'wb') as f:
            pickle.dump(G, f, pickle.HIGHEST_PROTOCOL)
        
        print(f"Test graph for hour {slice_index} saved to {graph_path}")

# Usage Example for graph creation
if __name__ == "__main__":
    # Read CSV and prepare DataFrame.
    df_test = pd.read_csv('filtered_train_3edge.csv')
    df_test['Timestamp'] = pd.to_datetime(df_test['Timestamp'])
    # Set Timestamp as index and sort (required for grouping by hour)
    df_test = df_test.set_index('Timestamp').sort_index()

    output_test_dir = "3ed_trai_h_graphs"
    create_test_graphs_edge_labels(df_test, output_test_dir)

Hour 0:
Label
0    6
Name: count, dtype: int64
Test graph for hour 0 saved to 3ed_trai_h_graphs/test_graph_hour_0.gpickle
Hour 1:
Label
0    71
Name: count, dtype: int64
Test graph for hour 1 saved to 3ed_trai_h_graphs/test_graph_hour_1.gpickle
Hour 2:
Label
0    84
Name: count, dtype: int64
Test graph for hour 2 saved to 3ed_trai_h_graphs/test_graph_hour_2.gpickle
Hour 3:
Label
0    78
Name: count, dtype: int64
Test graph for hour 3 saved to 3ed_trai_h_graphs/test_graph_hour_3.gpickle
Hour 4:
Label
0    41
Name: count, dtype: int64
Test graph for hour 4 saved to 3ed_trai_h_graphs/test_graph_hour_4.gpickle
Hour 653:
Label
0    62
Name: count, dtype: int64
Test graph for hour 653 saved to 3ed_trai_h_graphs/test_graph_hour_653.gpickle
Hour 654:
Label
0    125
Name: count, dtype: int64
Test graph for hour 654 saved to 3ed_trai_h_graphs/test_graph_hour_654.gpickle
Hour 655:
Label
0    146
Name: count, dtype: int64
Test graph for hour 655 saved to 3ed_trai_h_graphs/test_graph_hour_655.gpick

In [4]:
#########Replica of previous code:
import pandas as pd
import networkx as nx
import os
import pickle  # Added for saving graphs

def add_node_features(G):
    """
    Adds additional features to nodes in the graph, including:
    - Node degree
    - Community ID
    - Temporal activity (average edge count per node)
    - Node centrality (betweenness centrality)

    Parameters:
        G (nx.MultiDiGraph): The input graph.

    Returns:
        nx.MultiDiGraph: The graph with added node features.
    """
    # Add degree
    for node in G.nodes:
        G.nodes[node]['degree'] = G.degree[node]

    # Add community detection (Label Propagation)
    undirected_graph = nx.Graph(G)  # Convert to undirected for community detection
    communities = nx.community.label_propagation_communities(undirected_graph)
    community_mapping = {node: community_id for community_id, community in enumerate(communities) for node in community}
    for node in G.nodes:
        G.nodes[node]['community'] = community_mapping.get(node, -1)

    # Add centrality (Betweenness Centrality)
    centrality = nx.betweenness_centrality(G)
    for node, value in centrality.items():
        G.nodes[node]['centrality'] = value

    return G

def create_test_graphs_edge_labels(df, output_dir):
    """
    Split the DataFrame into hourly slices and create graphs for each slice.
    Each edge gets a valid label (e.g., 0 or 1) read from the DataFrame.
    
    Parameters:
        df (pd.DataFrame): The input DataFrame with temporal data.
        output_dir (str): Directory to save the graphs.
    """
    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)
    
    # Group the DataFrame into hourly slices using the datetime index.
    # Changed 'H' to 'h' to resolve the pandas FutureWarning
    time_slices = [g for _, g in df.groupby(pd.Grouper(freq='h'))]
    
    for slice_index, slice_df in enumerate(time_slices):
        if slice_df.empty:
            continue

        # Print value counts of the 'Label' column in this time-slice.
        print(f"Hour {slice_index}:")
        print(slice_df['Label'].value_counts())
        
        # Create a MultiDiGraph for this time-slice.
        G = nx.MultiDiGraph()

        for _, row in slice_df.iterrows():
            src_ip = row['Src IP']
            dst_ip = row['Dst IP']
            
            # Convert the label to an int (if missing or invalid, you can decide a fallback; here we assume it is valid)
            try:
                label = int(row['Label'])
            except Exception as e:
                print(f"Skipping row due to invalid label: {row['Label']}; error: {e}")
                continue

            if pd.isna(src_ip) or pd.isna(dst_ip):
                continue

            # Add nodes if not already present.
            if not G.has_node(src_ip):
                G.add_node(src_ip)
            if not G.has_node(dst_ip):
                G.add_node(dst_ip)

            # Add edges for different interactions.
            # 1. Network Edge (Communication stats)
            G.add_edge(src_ip, dst_ip, key='network', label=label,
                       src_port=row['Src Port'], dst_port=row['Dst Port'], 
                       protocol=row['Protocol'], pkt_len_mean=row['Pkt Len Mean'], 
                       pkt_len_max=row['Pkt Len Max'], pkt_len_var=row['Pkt Len Var'],
                       bwd_pkt_len_mean=row['Bwd Pkt Len Mean'], bwd_pkt_len_std=row['Bwd Pkt Len Std'],
                       bwd_pkt_len_max=row['Bwd Pkt Len Max'], fwd_pkt_len_mean=row['Fwd Pkt Len Mean'],
                       fwd_pkt_len_max=row['Fwd Pkt Len Max'], tot_len_fwd=row['TotLen Fwd Pkts'],
                       tot_len_bwd=row['TotLen Bwd Pkts'], down_up_ratio=row['Down/Up Ratio'],
                       interaction='network_communication')

            # 2. Context Edge (Timing and environment)
            G.add_edge(src_ip, dst_ip, key='context', label=label,
                       idle_min=row['Idle Min'], idle_max=row['Idle Max'], idle_mean=row['Idle Mean'],
                       fwd_seg_size_avg=row['Fwd Seg Size Avg'], fwd_seg_size_min=row['Fwd Seg Size Min'],
                       subflow_fwd_byts=row['Subflow Fwd Byts'],
                       interaction='context')

            # 3. Knowledge Edge (Protocol/Header rules)
            G.add_edge(src_ip, dst_ip, key='knowledge', label=label,
                       fwd_header_len=row['Fwd Header Len'], bwd_header_len=row['Bwd Header Len'],
                       init_fwd_win=row['Init Fwd Win Byts'], init_bwd_win=row['Init Bwd Win Byts'],
                       interaction='knowledge')

        # Add node features
        G = add_node_features(G)

        # Save the graph as a .gpickle file using standard pickle
        graph_path = os.path.join(output_dir, f"test_graph_hour_{slice_index}.gpickle")
        with open(graph_path, 'wb') as f:
            pickle.dump(G, f, pickle.HIGHEST_PROTOCOL)
            
        print(f"Test graph for hour {slice_index} saved to {graph_path}")

# Usage Example for graph creation
if __name__ == "__main__":
    # Read CSV and prepare DataFrame.
    df_test = pd.read_csv('filtered_train_3edge.csv')
    df_test['Timestamp'] = pd.to_datetime(df_test['Timestamp'])
    # Set Timestamp as index and sort (required for grouping by hour)
    df_test = df_test.set_index('Timestamp').sort_index()

    output_test_dir = "3ed_trai_h_graphs"
    create_test_graphs_edge_labels(df_test, output_test_dir)

Hour 0:
Label
0    6
Name: count, dtype: int64
Test graph for hour 0 saved to 3ed_trai_h_graphs/test_graph_hour_0.gpickle
Hour 1:
Label
0    71
Name: count, dtype: int64
Test graph for hour 1 saved to 3ed_trai_h_graphs/test_graph_hour_1.gpickle
Hour 2:
Label
0    84
Name: count, dtype: int64
Test graph for hour 2 saved to 3ed_trai_h_graphs/test_graph_hour_2.gpickle
Hour 3:
Label
0    78
Name: count, dtype: int64
Test graph for hour 3 saved to 3ed_trai_h_graphs/test_graph_hour_3.gpickle
Hour 4:
Label
0    41
Name: count, dtype: int64
Test graph for hour 4 saved to 3ed_trai_h_graphs/test_graph_hour_4.gpickle
Hour 653:
Label
0    62
Name: count, dtype: int64
Test graph for hour 653 saved to 3ed_trai_h_graphs/test_graph_hour_653.gpickle
Hour 654:
Label
0    125
Name: count, dtype: int64
Test graph for hour 654 saved to 3ed_trai_h_graphs/test_graph_hour_654.gpickle
Hour 655:
Label
0    146
Name: count, dtype: int64
Test graph for hour 655 saved to 3ed_trai_h_graphs/test_graph_hour_655.gpick

# Community detection for graphs and then update the graph with the label of community for each node

In [9]:
import networkx as nx
import os
import pickle  # Added for loading and saving graphs

def detect_and_label_communities_lpa(graph):
    """
    Perform community detection using the Label Propagation Algorithm (LPA) and label nodes with community IDs.
    Adds 'x' attribute based on the 'community' label.

    Parameters:
        graph (nx.MultiDiGraph): Input graph.

    Returns:
        graph (nx.MultiDiGraph): Updated graph with community labels and 'x' attributes.
    """
    # Convert MultiDiGraph to Graph (undirected graph for LPA)
    undirected_graph = nx.Graph(graph)

    # Perform community detection using LPA
    communities = nx.community.label_propagation_communities(undirected_graph)

    # Assign community labels to nodes and add 'x' attribute
    for community_id, community in enumerate(communities):
        for node in community:
            graph.nodes[node]['community'] = community_id
            graph.nodes[node]['x'] = [community_id]  # 'x' is a feature; wrap in a list for PyTorch Geometric compatibility

    return graph


def process_graphs_with_lpa(input_dir, output_dir):
    """
    Detect communities using LPA, update graphs with community labels, and add 'x' attribute.
    
    Parameters:
        input_dir (str): Directory containing input graphs.
        output_dir (str): Directory to save updated graphs.
    """
    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)

    # Process each graph file in the input directory
    for graph_file in os.listdir(input_dir):
        if not graph_file.endswith('.gpickle'):
            continue
        
        # Load the graph using standard pickle
        graph_path = os.path.join(input_dir, graph_file)
        with open(graph_path, 'rb') as f:
            G = pickle.load(f)

        # Detect communities using LPA and label nodes
        G = detect_and_label_communities_lpa(G)

        # Save the updated graph using standard pickle
        updated_graph_path = os.path.join(output_dir, graph_file)
        with open(updated_graph_path, 'wb') as f:
            pickle.dump(G, f, pickle.HIGHEST_PROTOCOL)
            
        print(f"Updated graph with LPA communities and 'x' attribute saved to {updated_graph_path}")


# Example usage
if __name__ == "__main__":
    # Input directory containing graphs
    input_graph_dir = "3ed_trai_h_graphs"

    # Output directory for updated graphs
    output_graph_dir = "3ed_trai_h_graphs_commun"

    # Process graphs and add community labels using LPA
    process_graphs_with_lpa(input_graph_dir, output_graph_dir)

Updated graph with LPA communities and 'x' attribute saved to 3ed_trai_h_graphs_commun/test_graph_hour_656.gpickle
Updated graph with LPA communities and 'x' attribute saved to 3ed_trai_h_graphs_commun/test_graph_hour_4.gpickle
Updated graph with LPA communities and 'x' attribute saved to 3ed_trai_h_graphs_commun/test_graph_hour_1403.gpickle
Updated graph with LPA communities and 'x' attribute saved to 3ed_trai_h_graphs_commun/test_graph_hour_1413.gpickle
Updated graph with LPA communities and 'x' attribute saved to 3ed_trai_h_graphs_commun/test_graph_hour_660.gpickle
Updated graph with LPA communities and 'x' attribute saved to 3ed_trai_h_graphs_commun/test_graph_hour_1397.gpickle
Updated graph with LPA communities and 'x' attribute saved to 3ed_trai_h_graphs_commun/test_graph_hour_1886.gpickle
Updated graph with LPA communities and 'x' attribute saved to 3ed_trai_h_graphs_commun/test_graph_hour_671.gpickle
Updated graph with LPA communities and 'x' attribute saved to 3ed_trai_h_graph

# convert Multigraph to hetrodata

In [10]:
import torch
from torch_geometric.data import HeteroData
import networkx as nx
import os
import pickle # Added for loading the gpickle files

def multiDiGraph_to_hetero_with_label(G: nx.MultiDiGraph) -> HeteroData:
    """
    Converts a MultiDiGraph with multiple edge types to a HeteroData object.
    Preserves the 'label' field in data[rel_type].edge_label.
    """
    data = HeteroData()
    node_mapping = {node: i for i, node in enumerate(G.nodes())}
    data['ip'].num_nodes = G.number_of_nodes()

    # Add node-level features
    x = []
    community_labels = []
    for node in G.nodes():
        community = G.nodes[node].get('community', -1)
        community_labels.append(community)
        x.append([community])
    data['ip'].community = torch.tensor(community_labels, dtype=torch.long)
    data['ip'].x = torch.tensor(x, dtype=torch.float)

    # Process each edge from G.
    for u, v, key, edge_attrs in G.edges(data=True, keys=True):
        src = node_mapping[u]
        dst = node_mapping[v]
        rel_type = ('ip', key, 'ip')
        if rel_type not in data.edge_types:
            data[rel_type].edge_index = []
            data[rel_type].edge_attr = []
            data[rel_type].edge_label = []  # Container for the label

        data[rel_type].edge_index.append([src, dst])
        feature_vec = []
        if key == 'network':
            for attr_name in ['src_port', 'dst_port', 'protocol', 'pkt_len_mean', 'pkt_len_max', 'pkt_len_var', 'bwd_pkt_len_mean', 'bwd_pkt_len_std', 'bwd_pkt_len_max', 'fwd_pkt_len_mean', 'fwd_pkt_len_max', 'tot_len_fwd', 'tot_len_bwd', 'down_up_ratio']:
                feature_vec.append(edge_attrs.get(attr_name, 0))
        elif key == 'context':
            for attr_name in ['idle_min', 'idle_max', 'idle_mean', 'fwd_seg_size_avg', 'fwd_seg_size_min', 'subflow_fwd_byts']:
                feature_vec.append(edge_attrs.get(attr_name, 0))
        elif key == 'knowledge':
            for attr_name in ['fwd_header_len', 'bwd_header_len', 'init_fwd_win', 'init_bwd_win']:
                feature_vec.append(edge_attrs.get(attr_name, 0))
        data[rel_type].edge_attr.append(feature_vec)
        # Save the label
        label = edge_attrs.get('label', -1)  # Default to -1 if label is missing
        if label == -1:
            print(f"Warning: Missing or invalid label for edge {u} -> {v} of type {key}")
        data[rel_type].edge_label.append(label)

    # Convert lists to tensors.
    for rel_type in data.edge_types:
        data[rel_type].edge_index = torch.tensor(data[rel_type].edge_index, dtype=torch.long).t().contiguous()
        if data[rel_type].edge_attr:
            data[rel_type].edge_attr = torch.tensor(data[rel_type].edge_attr, dtype=torch.float)
        if data[rel_type].edge_label:
            data[rel_type].edge_label = torch.tensor(data[rel_type].edge_label, dtype=torch.long)
    return data

def process_and_save_hetero_graphs_with_label(input_dir, output_dir):
    """
    Converts all .gpickle graphs in a directory to HeteroData objects and saves them as .pt,
    preserving the 'label' field in data[rel_type].edge_label.
    """
    os.makedirs(output_dir, exist_ok=True)
    for graph_file in os.listdir(input_dir):
        if not graph_file.endswith('.gpickle'):
            continue
        graph_path = os.path.join(input_dir, graph_file)
        
        # --- REPLACED NETWORKX GPICKLE WITH STANDARD PICKLE ---
        with open(graph_path, 'rb') as f:
            G = pickle.load(f)
            
        hetero_data = multiDiGraph_to_hetero_with_label(G)
        hetero_path = os.path.join(output_dir, graph_file.replace('.gpickle', '.pt'))
        torch.save(hetero_data, hetero_path)
        print(f"Saved HeteroData with labels to {hetero_path}")

if __name__ == "__main__":
    input_test_dir = "3ed_trai_h_graphs_commun"         # Input .gpickle files (with communities added)
    output_test_pt_dir = "3ed_trai_h_graphs_hetero_graphs" # Output .pt files
    process_and_save_hetero_graphs_with_label(input_test_dir, output_test_pt_dir)

Saved HeteroData with labels to 3ed_trai_h_graphs_hetero_graphs/test_graph_hour_656.pt
Saved HeteroData with labels to 3ed_trai_h_graphs_hetero_graphs/test_graph_hour_4.pt
Saved HeteroData with labels to 3ed_trai_h_graphs_hetero_graphs/test_graph_hour_1403.pt
Saved HeteroData with labels to 3ed_trai_h_graphs_hetero_graphs/test_graph_hour_1413.pt
Saved HeteroData with labels to 3ed_trai_h_graphs_hetero_graphs/test_graph_hour_660.pt
Saved HeteroData with labels to 3ed_trai_h_graphs_hetero_graphs/test_graph_hour_1397.pt
Saved HeteroData with labels to 3ed_trai_h_graphs_hetero_graphs/test_graph_hour_1886.pt
Saved HeteroData with labels to 3ed_trai_h_graphs_hetero_graphs/test_graph_hour_671.pt
Saved HeteroData with labels to 3ed_trai_h_graphs_hetero_graphs/test_graph_hour_1412.pt
Saved HeteroData with labels to 3ed_trai_h_graphs_hetero_graphs/test_graph_hour_1402.pt
Saved HeteroData with labels to 3ed_trai_h_graphs_hetero_graphs/test_graph_hour_1887.pt
Saved HeteroData with labels to 3ed_tr

# Test for inside of graph, no need to run it

In [11]:
#was test for inside of .pt ( no need to run)
import torch
import os

def inspect_pt_file(file_path):
    """
    Inspects the contents of a .pt file and prints its structure.

    Parameters:
        file_path (str): Path to the .pt file.
    """
    # Explicitly set weights_only=False because HeteroData is a custom Python object, 
    # suppressing the PyTorch Security FutureWarning.
    data = torch.load(file_path, weights_only=False)
    print(f"Inspecting file: {file_path}")
    print("-" * 40)

    # Check if it's a PyTorch Geometric HeteroData object
    if hasattr(data, 'keys') and hasattr(data, 'edge_index_dict'):
        print("File contains a HeteroData object.")
        print(f"Node types: {data.node_types}")
        for node_type in data.node_types:
            print(f"  Node type '{node_type}':")
            if 'x' in data[node_type]:
                print(f"    Node features 'x': shape {data[node_type].x.shape}")
            else:
                print("    No node features ('x') found.")
                
            # Check for the custom 'community' label we added
            if 'community' in data[node_type]:
                print(f"    Node 'community' labels: shape {data[node_type].community.shape}")
                
            if 'num_nodes' in data[node_type]:
                print(f"    Number of nodes: {data[node_type].num_nodes}")
        
        print(f"Edge types: {data.edge_types}")
        for edge_type in data.edge_types:
            print(f"  Edge type {edge_type}:")
            if 'edge_index' in data[edge_type]:
                print(f"    Edge index: shape {data[edge_type].edge_index.shape}")
            if 'edge_attr' in data[edge_type]:
                print(f"    Edge attributes: shape {data[edge_type].edge_attr.shape}")
            
            # Check for the custom 'edge_label' we preserved
            if 'edge_label' in data[edge_type]:
                print(f"    Edge labels: shape {data[edge_type].edge_label.shape}")
                
    elif isinstance(data, dict):
        print("File contains a dictionary. Keys:")
        for key, value in data.items():
            print(f"  {key}: {type(value)}")
            if isinstance(value, torch.Tensor):
                print(f"    Tensor shape: {value.shape}")
    else:
        print("Unknown data format.")
    print("-" * 40)

def inspect_all_pt_files(directory):
    """
    Inspects all .pt files in a given directory.

    Parameters:
        directory (str): Path to the directory containing .pt files.
    """
    print(f"Inspecting .pt files in directory: {directory}")
    for file in os.listdir(directory):
        if file.endswith(".pt"):
            inspect_pt_file(os.path.join(directory, file))

# Directory containing your .pt files
input_graph_dir = "3ed_trai_h_graphs_hetero_graphs"

# Inspect all files in the directory
inspect_all_pt_files(input_graph_dir)

Inspecting .pt files in directory: 3ed_trai_h_graphs_hetero_graphs
Inspecting file: 3ed_trai_h_graphs_hetero_graphs/test_graph_hour_675.pt
----------------------------------------
File contains a HeteroData object.
Node types: ['ip']
  Node type 'ip':
    Node features 'x': shape torch.Size([33, 1])
    Node 'community' labels: shape torch.Size([33])
    Number of nodes: 33
Edge types: [('ip', 'network', 'ip'), ('ip', 'context', 'ip'), ('ip', 'knowledge', 'ip')]
  Edge type ('ip', 'network', 'ip'):
    Edge index: shape torch.Size([2, 28])
    Edge attributes: shape torch.Size([28, 14])
    Edge labels: shape torch.Size([28])
  Edge type ('ip', 'context', 'ip'):
    Edge index: shape torch.Size([2, 28])
    Edge attributes: shape torch.Size([28, 6])
    Edge labels: shape torch.Size([28])
  Edge type ('ip', 'knowledge', 'ip'):
    Edge index: shape torch.Size([2, 28])
    Edge attributes: shape torch.Size([28, 4])
    Edge labels: shape torch.Size([28])
--------------------------------

In [12]:
#was test for inside of graph ( no need to run)
import os
import networkx as nx
import pickle  # Added for loading the graph

def inspect_community_in_gpickle(file_path):
    """
    Inspects the presence of the 'community' attribute in a .gpickle file.

    Parameters:
        file_path (str): Path to the .gpickle file.
    """
    print(f"Inspecting file: {file_path}")
    print("-" * 40)

    # Load the graph using standard pickle
    with open(file_path, 'rb') as f:
        G = pickle.load(f)

    # Check for 'community' attribute in nodes
    if all('community' in G.nodes[node] for node in G.nodes()):
        print(f"All nodes have a 'community' attribute.")
        print("Sample 'community' values:")
        sample_communities = {node: G.nodes[node]['community'] for node in list(G.nodes)[:10]}
        print(sample_communities)
    else:
        missing = [node for node in G.nodes() if 'community' not in G.nodes[node]]
        print(f"Some nodes are missing the 'community' attribute. Missing nodes: {missing[:10]} (only showing first 10)")

    print(f"Total nodes: {len(G.nodes())}")
    print("-" * 40)


def inspect_all_gpickle_files(directory):
    """
    Inspects the 'community' attribute in all .gpickle files in a given directory.

    Parameters:
        directory (str): Path to the directory containing .gpickle files.
    """
    print(f"Inspecting .gpickle files in directory: {directory}")
    for file in os.listdir(directory):
        if file.endswith(".gpickle"):
            inspect_community_in_gpickle(os.path.join(directory, file))


# Directory containing your .gpickle files
input_graph_dir = "3ed_trai_h_graphs_commun"

# Inspect all files in the directory for the 'community' attribute
inspect_all_gpickle_files(input_graph_dir)

Inspecting .gpickle files in directory: 3ed_trai_h_graphs_commun
Inspecting file: 3ed_trai_h_graphs_commun/test_graph_hour_656.gpickle
----------------------------------------
All nodes have a 'community' attribute.
Sample 'community' values:
{'132.211.192.168': 0, '1.152.192.168': 0, '41.238.224.122': 1, '192.168.1.190': 1, '133.81.192.168': 0, '249.83.3.122': 2, '49.24.192.168': 2, '131.123.192.168': 0, '128.183.192.168': 0, '127.246.192.168': 0}
Total nodes: 54
----------------------------------------
Inspecting file: 3ed_trai_h_graphs_commun/test_graph_hour_4.gpickle
----------------------------------------
All nodes have a 'community' attribute.
Sample 'community' values:
{'100.176.192.168': 0, '1.152.192.168': 0, '230.97.192.168': 0, '176.40.3.122': 1, '49.24.192.168': 1, '206.145.192.168': 2, '1.192.192.168': 2, '98.45.192.168': 0, '229.242.192.168': 0, '177.30.87.144': 3}
Total nodes: 48
----------------------------------------
Inspecting file: 3ed_trai_h_graphs_commun/test_gra